# EL(Extract and Load) Postgres AdventureWorks to MinIO

### Install python dotenv for get the environment variables

In [2]:
pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.


## Is important to avoid inconsistencies, the jupyer notebook container and apache spark container has the same pyspark version, in my case is 3.5.0

### Print version from pyspark, Put the same version from your apache spark

In [1]:
import pyspark

print(pyspark.__version__)

3.5.0


## Imports from pyspark, files and log

In [3]:
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession
from pyspark.sql.functions import date_format
import logging
from configurations import configurations as config_file # Import configurations.py from the configurations folder
from functions import functions as func_file # Import functions.py from the functions folder
from datetime import datetime

# Import for get the environment variables 
from dotenv import load_dotenv
import os

## Import environment variables

In [4]:
load_dotenv()

MINIO_CONTAINER=os.getenv("MINIO_CONTAINER")
MINIO_USER=os.getenv("MINIO_USER")
MINIO_PASSWORD=os.getenv("MINIO_PASSWORD")
POSTGRES_CONTAINER=os.getenv("POSTGRES_CONTAINER")
POSTGRES_USER=os.getenv("POSTGRES_USER")
POSTGRES_PASSWORD=os.getenv("POSTGRES_PASSWORD")

## Configuration from Spark

In [5]:
conf = SparkConf()

conf.setAppName("Extract and Load postgres adventureworks to minIO") # Spark application name, Usefull for logs
# Add the jars from hadoop-aws and aws-java-sdk-bundle is necessary for org.apache.hadoop.fs.s3a.S3AFileSystem,
# add the Postgresql JDBC jar is necessary for connect on database. Add the delta-spark is necessary for delta catalog, all this Jars is auto-download from spark
conf.set("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,"
         "com.amazonaws:aws-java-sdk-bundle:1.12.767,"
         "org.postgresql:postgresql:42.7.2,"
         "io.delta:delta-spark_2.12:3.2.0")
conf.set("spark.hadoop.fs.s3a.endpoint",f"http://{MINIO_CONTAINER}:9000") # Container and Port from MinIO
conf.set("spark.hadoop.fs.s3a.access.key", MINIO_USER) # Login from MinIO
conf.set("spark.hadoop.fs.s3a.secret.key", MINIO_PASSWORD) # Password from MinIO
conf.set("spark.hadoop.fs.s3a.path.style.access", True) # Enforces the use of URLs as the format. Without this, Spark attempts to use the AWS standard (bucket.endpoint), which fails in MinIO
conf.set("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") # Talk to Hadoop/Spark to use new conector S3A
conf.set("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") # How to credentials are acess via config(access key + secret)
conf.set("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") # active extension from Delta Lake
conf.set("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") # Change the standard catalog from spark to Delta 
conf.set("hive.metastore.uris", "thrift://metastore:9083") # Connect to Hive Metastore external

spark = SparkSession.builder.config(conf=conf).enableHiveSupport().getOrCreate()

## Configuration to activate the logging

In [6]:
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

# Logging the Start process from ingestion
logging.info("Starting ingestions from Postgres adventureworks to MinIO landing...")

2026-04-30 20:36:16,976 - INFO - Starting ingestions from Postgres adventureworks to MinIO landing...


## Reading each table from postges and writing on minIO landing

In [7]:
for table_input_name in config_file.tables_postgres_adventureworks.values():
    
    try:
        # convert the table name from postgres to name in minIO s3
        table_input_path_s3 = func_file.convert_table_name(table_input_name)

        # Reading table in postgres via JDBC
        df_input_data = spark.read \
            .format("jdbc") \
            .option("url", f"jdbc:postgresql://{POSTGRES_CONTAINER}:5432/Adventureworks") \
            .option("dbtable", table_input_name) \
            .option("user", POSTGRES_USER) \
            .option("password", POSTGRES_PASSWORD) \
            .option("driver", "org.postgresql.Driver") \
            .load()

        # Landing path
        landing_path = config_file.data_lakehouse_path["landing"]
        output_table_path = f"{landing_path}{table_input_path_s3}"

        # Logging the processing
        logging.info(f"processing table {table_input_path_s3}")

        # Adding a new column data related the load data
        df_with_update_data = func_file.add_data_last_update(df_input_data)

        # modifing the column "modifieddate" from postgres tables for add a new column "month_key" to create a partition on the minIO landing
        df_with_month_partition = df_with_update_data.withColumn("month_key", date_format(df_with_update_data["modifieddate"], "yyyy-MM"))
        
        # Writing the dataframe on minIO landing
        df_with_month_partition.write.format("parquet").mode("overwrite").partitionBy("month_key").save(output_table_path)
        
        # Logging the sucessfully process
        logging.info(f"Table {table_input_path_s3} Sucessfully processed and saved in MinIO landing on: {output_table_path}")

    
    except Exception as e:
        # Logging the Error
         logging.error(f"Error processing table {table_input_name}: {str(e)}")

    # Logging the Ingestion
    logging.info(f"Ingestions to landing completed!")
    

2026-04-30 20:36:46,540 - INFO - processing table sales_countryregioncurrency
2026-04-30 20:36:56,224 - INFO - Table sales_countryregioncurrency Sucessfully processed and saved in MinIO landing on: s3a://landing/adventureworks/sales_countryregioncurrency
2026-04-30 20:36:56,226 - INFO - Ingestions to landing completed!
